# IR System — TF-IDF Retrieval
**Step 4:** Build TF-IDF index and run retrieval for both datasets.

Uses sklearn sparse matrix for fast cosine similarity (seconds, not hours).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
SAVE_DIR = '/content/drive/MyDrive/ir_system_data'
import os, sys

if not os.path.exists('/content/ir-system'):
    !git clone https://github.com/ghazal-mohammad/ir-system.git /content/ir-system
else:
    !cd /content/ir-system && git pull
sys.path.insert(0, '/content/ir-system')

!pip install ir-datasets==0.5.9 scikit-learn -q
print('ready')

In [ ]:
import json
from services.tfidf_service import (
    build_tfidf_matrix, retrieve_tfidf_fast,
    save_tfidf_matrix, load_tfidf_matrix
)
from services.preprocessing_service import preprocess_to_string, preprocess

# Load processed docs
print('Loading CT2021 docs...')
with open(f'{SAVE_DIR}/ct2021_docs_processed.json') as f:
    docs1 = json.load(f)
print(f'CT2021: {len(docs1):,} docs')

print('Loading MSMARCO docs...')
with open(f'{SAVE_DIR}/msmarco_docs_processed.json') as f:
    docs2 = json.load(f)
print(f'MSMARCO: {len(docs2):,} docs')

In [ ]:
# Build TF-IDF sparse matrix for CT2021 (~2-3 min)
print('Building TF-IDF matrix for CT2021...')
vec1, mat1, doc_ids1 = build_tfidf_matrix(docs1)
print(f'Matrix shape: {mat1.shape}  (docs x terms)')
save_tfidf_matrix(vec1, mat1, doc_ids1, f'{SAVE_DIR}/ct2021')
print('CT2021 done')

In [ ]:
# Build TF-IDF sparse matrix for MSMARCO (~3-4 min)
print('Building TF-IDF matrix for MSMARCO...')
vec2, mat2, doc_ids2 = build_tfidf_matrix(docs2)
print(f'Matrix shape: {mat2.shape}')
save_tfidf_matrix(vec2, mat2, doc_ids2, f'{SAVE_DIR}/msmarco')
print('MSMARCO done')

In [ ]:
# Load queries
import ir_datasets

ds1 = ir_datasets.load('clinicaltrials/2021/trec-ct-2021')
queries1 = {q.query_id: q.text for q in ds1.queries_iter()}

ds2 = ir_datasets.load('msmarco-passage/trec-dl-2019')
queries2 = {q.query_id: q.text for q in ds2.queries_iter()}

print(f'CT2021: {len(queries1)} queries')
print(f'MSMARCO: {len(queries2)} queries')

In [ ]:
# Quick test
sample_q = list(queries1.values())[0]
results = retrieve_tfidf_fast(sample_q, vec1, mat1, doc_ids1, top_k=10)
print(f'Query: "{sample_q}"')
print('Top 10:')
for r in results:
    print(f'  rank {r["rank"]}: {r["doc_id"]} (score={r["score"]})')

In [ ]:
# Full retrieval — CT2021 (should finish in < 1 minute)
import time
print('Running TF-IDF on all CT2021 queries...')
t0 = time.time()
all_results1 = {}
for qid, qtext in queries1.items():
    all_results1[qid] = retrieve_tfidf_fast(qtext, vec1, mat1, doc_ids1, top_k=1000)
print(f'Done in {time.time()-t0:.1f}s — {len(all_results1)} queries')

with open(f'{SAVE_DIR}/ct2021_tfidf_results.json', 'w') as f:
    json.dump(all_results1, f)
print('CT2021 results saved')

In [ ]:
# Full retrieval — MSMARCO
print('Running TF-IDF on all MSMARCO queries...')
t0 = time.time()
all_results2 = {}
for qid, qtext in queries2.items():
    all_results2[qid] = retrieve_tfidf_fast(qtext, vec2, mat2, doc_ids2, top_k=1000)
print(f'Done in {time.time()-t0:.1f}s — {len(all_results2)} queries')

with open(f'{SAVE_DIR}/msmarco_tfidf_results.json', 'w') as f:
    json.dump(all_results2, f)
print('MSMARCO results saved')

print('\n=== TF-IDF Retrieval Complete ===')
print('Next: 05_retrieval_bm25.ipynb')